# Benchmark — Google Colab

Pipeline:
1. Mount Google Drive & giải nén code
2. Cài dependencies
3. Kiểm tra GPU & dữ liệu
4. Chạy thí nghiệm (single experiment hoặc full ablation)

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Giải nén code

**Yêu cầu:** Upload file `code.zip` (chứa toàn bộ thư mục `code/`) lên Google Drive trước.

Cấu trúc zip cần có:
```
code/
  benchmark/
  embedding_generator/
  profile_generator/
```

In [ ]:
import zipfile, os

# Sửa đường dẫn nếu bạn để code.zip ở thư mục khác trong Drive
ZIP_PATH = '/content/drive/MyDrive/code.zip'
EXTRACT_PATH = '/content'

if not os.path.exists('/content/code'):
    print('Đang giải nén...')
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall(EXTRACT_PATH)
    print('Xong!')
else:
    print('Code đã được giải nén rồi.')

os.listdir('/content/code')

## 3. Cài đặt dependencies

In [ ]:
%pip install -q torch numpy pandas scipy scikit-learn sentence-transformers tqdm pyyaml

## 4. Kiểm tra GPU & môi trường

In [ ]:
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('Không có GPU — vào Runtime > Change runtime type > GPU để bật')

## 5. Kiểm tra dữ liệu & embeddings

In [ ]:
from pathlib import Path

CODE_ROOT = Path('/content/code')
BENCHMARK_DIR = CODE_ROOT / 'benchmark'

# Kiểm tra dữ liệu ML-20M
ml20m_dir = CODE_ROOT / 'profile_generator' / 'llm-movie-profiler-v1-20260402' / 'data' / 'ml-20m'
required_data = ['ratings.csv', 'movies.csv', 'genome-scores.csv', 'genome-tags.csv']
print('=== Dữ liệu ML-20M ===')
for f in required_data:
    path = ml20m_dir / f
    status = 'OK' if path.exists() else 'THIẾU'
    print(f'  [{status}] {f}')

# Kiểm tra embeddings
emb_dir = CODE_ROOT / 'embedding_generator' / 'output' / 'bge-large-v1.5'
required_emb = [
    'profile_embeddings.npy',
    'mood_vectors.npy',
    'theme_matrix.npy',
    'genome_embeddings.npy',
    'movie_id_index.json',
]
print('\n=== Embeddings ===')
for f in required_emb:
    path = emb_dir / f
    status = 'OK' if path.exists() else 'THIẾU'
    print(f'  [{status}] {f}')

## 6. Thêm benchmark vào Python path

In [ ]:
import sys
sys.path.insert(0, str(BENCHMARK_DIR))

# Kiểm tra import
from config import EMBED_DIM, NUM_EPOCHS, PATIENCE, FEATURE_CONFIGS
print(f'EMBED_DIM={EMBED_DIM}, NUM_EPOCHS={NUM_EPOCHS}, PATIENCE={PATIENCE}')
print(f'Feature configs: {list(FEATURE_CONFIGS.keys())}')

## 7. Chạy một thí nghiệm đơn

Chỉnh các tham số bên dưới rồi chạy cell.

| Tham số | Các lựa chọn |
|---|---|
| `MODEL` | `bpr_mf`, `lightgcn`, `lightgcn_sf`, `xsimgcl`, `simgcl`, `lightgcl`, `kar` |
| `FEATURES` | `none`, `genome`, `bert_title`, `llm_profile`, `llm_mood`, `llm_themes`, `llm_prof_mood`, `llm_all`, `genome_llm` |
| `SEED` | Bất kỳ số nguyên nào (mặc định: 42) |

In [ ]:
import os
os.chdir(str(BENCHMARK_DIR))

MODEL    = 'lightgcn'   # chỉnh ở đây
FEATURES = 'none'       # chỉnh ở đây
SEED     = 42           # chỉnh ở đây
EPOCHS   = 200          # giảm xuống 20-50 để test nhanh

!python run_experiment.py \
    --model {MODEL} \
    --features {FEATURES} \
    --seed {SEED} \
    --epochs {EPOCHS} \
    --device auto

## 8. Chạy Quick Test (tất cả model, 1 seed, 20 epochs)

Dùng để kiểm tra nhanh toàn bộ pipeline trước khi chạy full.

In [ ]:
os.chdir(str(BENCHMARK_DIR))
!python run_ablation.py --quick

## 9. Chạy R1 — RLMRec-plus (Contrastive distillation)

Chỉnh `SEED` rồi chạy cell.

In [ ]:
import os

RLMREC_DIR = '/content/code/benchmark/external/RLMRec'
os.chdir(RLMREC_DIR)

SEED   = 42   # chỉnh ở đây
EPOCHS = 100  # chỉnh ở đây

!python encoder/train_encoder.py \
    --model lightgcn_plus \
    --dataset ml20m \
    --device cuda \
    --seed {SEED} \
    --epochs {EPOCHS}

## 10. Chạy R2 — RLMRec-gene (Generative reconstruction)

Chỉnh `SEED` rồi chạy cell.

In [ ]:
import os

RLMREC_DIR = '/content/code/benchmark/external/RLMRec'
os.chdir(RLMREC_DIR)

SEED   = 42   # chỉnh ở đây
EPOCHS = 100  # chỉnh ở đây

!python encoder/train_encoder.py \
    --model lightgcn_gene \
    --dataset ml20m \
    --device cuda \
    --seed {SEED} \
    --epochs {EPOCHS}

## 11. Lưu checkpoint lên Google Drive (thủ công)

Chạy cell này bất cứ lúc nào để sao lưu `checkpoints/` và `results/` lên Drive.

In [ ]:
DRIVE_SAVE_DIR = '/content/drive/MyDrive/benchmark_checkpoints'

# Kiểm tra Drive đã mount chưa
import os
if not os.path.exists('/content/drive/MyDrive'):
    print('[LỖI] Drive chưa được mount — hãy chạy cell 1 trước.')
else:
    !mkdir -p "{DRIVE_SAVE_DIR}"
    
    if os.path.exists('/content/code/benchmark/checkpoints'):
        print('Đang lưu checkpoints/...')
        !cp -r /content/code/benchmark/checkpoints "{DRIVE_SAVE_DIR}/"
        print('[OK] checkpoints/ đã lưu')
    else:
        print('[SKIP] checkpoints/ chưa tồn tại')

    if os.path.exists('/content/code/benchmark/results'):
        print('Đang lưu results/...')
        !cp -r /content/code/benchmark/results "{DRIVE_SAVE_DIR}/"
        print('[OK] results/ đã lưu')
    else:
        print('[SKIP] results/ chưa tồn tại')

    print(f'Xong → {DRIVE_SAVE_DIR}')
